In [1]:
#Inicialización 
import xrfclk
import xrfdc
import pynq
from pynq import Overlay, MMIO
from pynq import lib
import numpy as np
import time
import os
import subprocess
import time 

CLOCKWIZARD_LOCK_ADDRESS = 0x0004
CLOCKWIZARD_RESET_ADDRESS = 0x0000
CLOCKWIZARD_RESET_TOKEN = 0x000A
MTS_START_TILE = 0x01
MAX_DAC_TILES = 4
MAX_ADC_TILES = 4
DAC_REF_TILE = 2
ADC_REF_TILE = 2

RFSOC4X2_LMK_FREQ = 500.0
RFSOC4X2_LMX_FREQ = 500.0
RFSOC4X2_DAC_TILES = 0b0101
RFSOC4X2_ADC_TILES = 0b0101

In [2]:
xrfclk.set_ref_clks(lmk_freq = RFSOC4X2_LMK_FREQ, lmx_freq = RFSOC4X2_LMX_FREQ) # Cargar la configuración deseada
#xrfclk.set_ref_clks(lmk_freq = 245.76, lmx_freq = 491.52) # Cargar la configuración base 

In [3]:
# Comprobación de que no haya un Overlay ya cargado
board = os.getenv('BOARD') 
# Run lsmod command to get the loaded modules list
output = subprocess.check_output(['lsmod'])
# Check if "zocl" is present in the output
if b'zocl' in output:
    # If present, remove the module using rmmod command
    rmmod_output = subprocess.run(['rmmod', 'zocl'])
    # Check return code
    assert rmmod_output.returncode == 0, "Could not restart zocl. Please Shutdown All Kernels and then restart"
    # If successful, load the module using modprobe command
    modprobe_output = subprocess.run(['modprobe', 'zocl'])
    assert modprobe_output.returncode == 0, "Could not restart zocl. It did not restart as expected"
else:
    modprobe_output = subprocess.run(['modprobe', 'zocl'])
    # Check return code
    assert modprobe_output.returncode == 0, "Could not restart ZOCL!"

In [4]:
# Cargamos el Overlay
ol = Overlay('/usr/local/share/pynq-venv/lib/python3.10/site-packages/pynq/overlays/Sin/design_1.bit',ignore_version=True)

In [5]:
# Sincronización de los convertidores 
ol.ACTIVE_DAC_TILES = RFSOC4X2_DAC_TILES
ol.ACTIVE_ADC_TILES = RFSOC4X2_ADC_TILES

ol.xrfdc = ol.usp_rf_data_converter_0
ol.xrfdc.mts_dac_config.RefTile = DAC_REF_TILE  # DAC tile distributing reference clock
ol.xrfdc.mts_adc_config.RefTile = ADC_REF_TILE  # ADC 

#INIT SYNC TILES
ol.xrfdc.mts_dac_config.Tiles = 0b0001 # turn only one tile on first
ol.xrfdc.mts_adc_config.Tiles = 0b0001
ol.xrfdc.mts_dac_config.SysRef_Enable = 1
ol.xrfdc.mts_adc_config.SysRef_Enable = 1
ol.xrfdc.mts_dac_config.Target_Latency = -1
ol.xrfdc.mts_adc_config.Target_Latency = -1

ol.xrfdc.mts_adc()
ol.xrfdc.mts_dac()

ol.ClockTree.clk_wiz_0.mmio.write_reg(CLOCKWIZARD_RESET_ADDRESS, CLOCKWIZARD_RESET_TOKEN)
time.sleep(0.1)
# Reset only user selected DAC tiles
bitvector = ol.ACTIVE_DAC_TILES
for n in range(MAX_DAC_TILES):
    if (bitvector & 0x1):
        ol.xrfdc.dac_tiles[n].Reset()
    bitvector = bitvector >> 1
# Reset ADC FIFO of only user selected tiles - restarts MTS engine
for toggleValue in range(0,1):
    bitvector = ol.ACTIVE_ADC_TILES
    for n in range(MAX_ADC_TILES):
        if (bitvector & 0x1):
            ol.xrfdc.adc_tiles[n].SetupFIFOBoth(toggleValue)
        bitvector = bitvector >> 1
#SYNC TILES
dacTarget=-1
adcTarget=-1
if ol.ACTIVE_DAC_TILES > 0:
    ol.xrfdc.mts_dac_config.Tiles = ol.ACTIVE_DAC_TILES # group defined in binary 0b1111
    ol.xrfdc.mts_dac_config.SysRef_Enable = 1
    ol.xrfdc.mts_dac_config.Target_Latency = dacTarget 
    ol.xrfdc.mts_dac()
else:
    ol.xrfdc.mts_dac_config.Tiles = 0x0
    ol.xrfdc.mts_dac_config.SysRef_Enable = 0
if ol.ACTIVE_ADC_TILES > 0:
    ol.xrfdc.mts_adc_config.Tiles = ol.ACTIVE_ADC_TILES
    ol.xrfdc.mts_adc_config.SysRef_Enable = 1
    ol.xrfdc.mts_adc_config.Target_Latency = adcTarget
    ol.xrfdc.mts_adc()
else:
    ol.xrfdc.mts_adc_config.Tiles = 0x0
    ol.xrfdc.mts_adc_config.SysRef_Enable = 0

In [6]:
# Instanciar el generador de señales 
DDFS=ol.Fuentes_de_Datos.Muestras_DDFS_Frec_c_0

In [7]:
# Definimos la frecuencia del tono 
DDFS.write(0x0,400)

In [8]:
# Instanciar los multiplexores de las secuencias 
Mux0=ol.Fuentes_de_Datos.Multiplexor_0
Mux1=ol.Fuentes_de_Datos.Multiplexor_1

In [9]:
# Seleccionar las entradas de datos 
select=2
Mux0.write(0x0,select)
Mux1.write(0x0,select)

In [2]:
# Selecciona entre la señal de calibración y la fuente de datos 
Mux3=ol.Mux_Cal_0
Mux3.write(0x0,1)

NameError: name 'ol' is not defined

In [10]:
# Instanciar la IP de lectura del GPIO 
GPIO=ol.axi_gpio_0

In [3]:
# Se lee la posición de los switches
print(GPIO.read(0x0))

NameError: name 'GPIO' is not defined